# Step 5 — Citations received by calendar year (resumable collection)

## Definition
For author a and year y, count distinct **(citing work, cited portfolio work)** links where the citing work's publication year is y. A paper citing two different papers by the same author contributes TWO citations. A repeated link contributes one. Co-authored target papers count for each author. Self-citations are included here, to be classified in step 6.

Count citations to **all known portfolio works**, not only works published in the ±5-year window. Bin by the CITING paper's year. Award year is separate from pre/post totals. These are retrospective citations in the current indexed graph, not a reconstruction of what OpenAlex knew in the historical year.

## Why collect historical edges?
The old works cache has lifetime cited_by_count, not yearly citation histories. Summary counts_by_year cannot safely establish all historical zeros; documented citation rollups exclude early history (before 2012). This notebook reconstructs all requested years uniformly from citation edges, rather than interpreting absent rollup years as zeros.

## Stages
1. Verify inputs, inventory old citation files, identify missing historical author-years.
2. Refresh portfolios for authors flagged in step 4 (125 in the reviewed output), recording proper pagination completion. Keep unchanged legacy portfolios explicitly unverified.
3. Batch up to 100 unique cited-work IDs per query; retrieve citing works and intersect their references with the batch. This preserves citation LINKS rather than undercounting with a union count of citing papers.
4. Resume collection with SQLite checkpoints. Export one row per author plus 11 author-year rows on EVERY run, even if collection is incomplete. Main counts stay missing until all that author's planned citation batches are complete.

## Credentials and budget
The key is NEVER embedded in this notebook. Set OPENALEX_API_KEY locally or enter it in a masked prompt. No keys, request headers, or authenticated URLs are written to disk or outputs. Because a key was shared in chat, rotate it before long-term use; never commit credentials.

Default cap: 200 HTTP attempts per run (roughly $0.02 at the quoted 1-credit list price), 5,000 attempts per UTC day through THIS collector. These are local limits, not visibility into spending elsewhere; check your OpenAlex dashboard first. Stop on 429 rather than automatically drawing extra requests. Budget and paging costs may change. Full collection can take multiple runs/days and considerable disk space; it is NOT promised to fit in one free day.

Run inside your repository, e.g. from `notebooks/05_yearly_citations.ipynb`. Existing upstream outputs remain unchanged. Live collection runs only on your machine, not when this notebook is delivered.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import getpass
import hashlib
import io
import json
import math
import os
import pickle
import re
import sqlite3
import time
import pandas as pd
import requests
from IPython.display import display

ROOT_OVERRIDE = None
STEP4_REL = 'conf_paper/conf_data/derived/yearly_publications_v1'
STEP2_REL = 'conf_paper/conf_data/derived/reference_awards_v1'
CACHE_REL = 'conf_paper/conf_data/04_all_works_cache.pkl'
COLLECTION_VERSION = 'yearly_citations_v1'
RUN_API_COLLECTION = True
MAX_REQUESTS_PER_RUN = 200
MAX_REQUESTS_PER_UTC_DAY = 5000
PER_PAGE = 100
BATCH_SIZE = 100
REQUEST_INTERVAL_SECONDS = 0.2
CORPUS = 'all'
TRANSFORM_VERSION = 'yearly_citations_1.0'
OFFSETS = list(range(-5, 6))

def now(): return datetime.now(timezone.utc).isoformat()
def day(): return datetime.now(timezone.utc).date().isoformat()
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def stable_hash(v): return hashlib.sha256(json.dumps(v, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
def csv(p): return pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8-sig')
def nid(v, prefix='W'):
    if not isinstance(v, str): return None
    m = re.fullmatch(r'(?:https?://openalex\.org/)?(' + prefix + r'\d+)/?', v.strip(), re.I)
    return m.group(1).upper() if m else None

def yr(v):
    if isinstance(v, bool): return None
    try:
        n = float(v)
        return int(n) if math.isfinite(n) and n.is_integer() and 1 <= n <= datetime.now(timezone.utc).year else None
    except (ValueError, TypeError, OverflowError): return None

def verify(folder):
    m = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    for name, h in m['output_sha256'].items():
        if not (folder / name).is_file() or sha(folder / name) != h:
            raise RuntimeError('Input integrity failure: ' + str(folder / name))
    return m

def root_dir():
    candidates = [Path(ROOT_OVERRIDE).expanduser().resolve()] if ROOT_OVERRIDE else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for p in candidates:
        if (p / STEP4_REL / 'manifest.json').is_file(): return p
    raise FileNotFoundError('Run inside the repository or set ROOT_OVERRIDE.')

ROOT = root_dir(); STEP4 = ROOT / STEP4_REL; STEP2 = ROOT / STEP2_REL
m4 = verify(STEP4); m2 = verify(STEP2)
if sha(STEP2 / 'manifest.json') != m4['identity']['input_hashes']['step2_manifest']:
    raise RuntimeError('Step 4 is linked to a different step-2 manifest.')
master4 = csv(STEP4 / 'author_yearly_publications.csv')
links = csv(STEP2 / 'reference_year_award_links.csv')
review4 = csv(STEP4 / 'cache_review.csv')
assert master4.author_id.is_unique
master4['reference_award_year'] = master4.reference_award_year.map(yr)
assert master4.reference_award_year.notna().all()
assert master4.reference_award_year.between(2000, 2018).all()
WINDOW_LO = int(master4.reference_award_year.min()) - 5
WINDOW_HI = int(master4.reference_award_year.max()) + 5

class BuiltinsOnlyUnpickler(pickle.Unpickler):
    def find_class(self, module, name): raise pickle.UnpicklingError('Global/class loading forbidden.')
    def persistent_load(self, pid): raise pickle.UnpicklingError('Persistent loading forbidden.')
cache_path = ROOT / CACHE_REL
if sha(cache_path) != m4['identity']['input_hashes']['works_cache']:
    raise RuntimeError('Works cache changed since step 4.')
cache = BuiltinsOnlyUnpickler(io.BytesIO(cache_path.read_bytes())).load()
if not isinstance(cache, dict) or not isinstance(cache.get('all_works'), dict):
    raise ValueError('Unexpected cache schema.')
legacy = defaultdict(set)
legacy_bad = set()
for aid_raw, works in cache['all_works'].items():
    aid = nid(aid_raw, 'A')
    if not aid: continue
    if not isinstance(works, list): legacy_bad.add(aid); continue
    for w in works:
        if isinstance(w, dict) and nid(w.get('id')):
            legacy[aid].add(nid(w['id']))  # KEEP target IDs even if their publication year is unknown.
        else: legacy_bad.add(aid)
expected_refs = defaultdict(set)
for r in links.itertuples(index=False):
    wid = nid(r.openalex_work_id)
    if wid: expected_refs[r.author_id].add(wid)
refresh_ids = set(review4.author_id) | legacy_bad | (set(master4.author_id) - set(legacy))
refresh_ids &= set(master4.author_id)
inputs = {'step4_manifest': sha(STEP4 / 'manifest.json'), 'step2_manifest': sha(STEP2 / 'manifest.json'), 'works_cache': sha(cache_path)}
print('Authors:', len(master4), '| Calendar-year window:', WINDOW_LO, 'through', WINDOW_HI)
print('Author portfolios queued for refresh:', len(refresh_ids))

Authors: 2710 | Calendar-year window: 1995 through 2023
Author portfolios queued for refresh: 125


## 1. Inventory and missing-history report

Old award-paper edge files are not assumed to cover all works of every award author. They are inventoried, not blindly treated as complete or used to establish zeros. The new collection is shared across all authors' target works, so co-authored targets are fetched once.

In [2]:
inventory_rows = []
for relative in ['data/matched/award_to_citing_edges.csv', 'data/matched/citing_edges_checkpoint.csv',
                 'data/matched/award_paper_citations.csv', 'data/matched/award_to_cited_edges.csv']:
    p = ROOT / relative
    row = {'path': relative, 'exists': p.is_file(), 'bytes': p.stat().st_size if p.is_file() else None,
           'used_as_complete_author_history': False}
    if p.is_file():
        try: row['columns_json'] = json.dumps(pd.read_csv(p, nrows=0).columns.tolist())
        except Exception: row['columns_json'] = 'unreadable_header'
    inventory_rows.append(row)
inventory = pd.DataFrame(inventory_rows)
history_need = pd.DataFrame([{'author_id': r.author_id, 'relative_year': k,
    'calendar_year': int(r.reference_award_year) + k,
    'historical_rollup_unsafe': int(r.reference_award_year) + k < 2012,
    'source_needed': 'citing_work_reference_edges_binned_by_citing_publication_year'}
    for r in master4.itertuples(index=False) for k in OFFSETS])
display(inventory)
print('Author-year cells to reconstruct:', len(history_need))
print('Cells before 2012 (must not treat rollup zeros as evidence):', int(history_need.historical_rollup_unsafe.sum()))

,path,exists,bytes,used_as_complete_author_history,columns_json
0,data/matched/award_to_citing_edges.csv,True,39095853,False,"[""award_paper_id"", ""award_year"", ""award_confer..."
1,data/matched/citing_edges_checkpoint.csv,True,39095853,False,"[""award_paper_id"", ""award_year"", ""award_confer..."
2,data/matched/award_paper_citations.csv,True,25198,False,"[""conference"", ""award_year"", ""work_id"", ""cited..."
3,data/matched/award_to_cited_edges.csv,True,2645002,False,"[""award_paper_id"", ""award_year"", ""award_confer..."


Author-year cells to reconstruct: 29810
Cells before 2012 (must not treat rollup zeros as evidence): 13085


## 2. Checkpoint database and strict page validation

SQLite stores query progress, distinct result IDs, citation edges and a local request ledger. Each page commits atomically. Restarting resumes the saved cursor without recounting edges. If the API's matched-record total changes during a query, or pages disagree or contain malformed records, the query is blocked for review rather than marked complete.

Keep the checkpoint database on your computer. It may become large; commit notebook and CSV reports, not credentials. Never delete the database to retry a budget stop.

In [3]:
STATE_DIR = ROOT / 'conf_paper/conf_data/collection' / COLLECTION_VERSION
STATE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = STATE_DIR / 'citation_collection.sqlite'
if 'conn' in globals():
    try: conn.close()
    except Exception: pass
conn = sqlite3.connect(DB_PATH, timeout=30)
conn.row_factory = sqlite3.Row
conn.executescript("""
CREATE TABLE IF NOT EXISTS metadata(key TEXT PRIMARY KEY,value TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS tasks(
 id TEXT PRIMARY KEY,kind TEXT NOT NULL,payload TEXT NOT NULL,status TEXT NOT NULL DEFAULT 'pending',
 cursor TEXT NOT NULL DEFAULT '*',expected INTEGER,pages INTEGER NOT NULL DEFAULT 0,
 issue TEXT NOT NULL DEFAULT '',started TEXT,finished TEXT);
CREATE TABLE IF NOT EXISTS items(
 task TEXT NOT NULL,id TEXT NOT NULL,year INTEGER,author_ids TEXT NOT NULL,
 PRIMARY KEY(task,id));
CREATE INDEX IF NOT EXISTS items_id_year ON items(id,year);
CREATE TABLE IF NOT EXISTS edges(
 task TEXT NOT NULL,citing TEXT NOT NULL,cited TEXT NOT NULL,year INTEGER NOT NULL,
 PRIMARY KEY(task,citing,cited));
CREATE INDEX IF NOT EXISTS edge_cited_year ON edges(cited,year);
CREATE TABLE IF NOT EXISTS request_ledger(utc_day TEXT NOT NULL,at TEXT NOT NULL,status TEXT);
""")
identity = {'inputs': inputs, 'transform_version': TRANSFORM_VERSION, 'corpus': CORPUS,
            'batch_size': BATCH_SIZE, 'per_page': PER_PAGE, 'window': [WINDOW_LO, WINDOW_HI]}
prior = conn.execute("SELECT value FROM metadata WHERE key='identity'").fetchone()
if prior and json.loads(prior['value']) != identity:
    raise RuntimeError('Checkpoint belongs to different inputs/settings. Use a new COLLECTION_VERSION after review.')
with conn:
    conn.execute("INSERT OR IGNORE INTO metadata VALUES('identity',?)", (json.dumps(identity, sort_keys=True),))

def add_task(kind, payload):
    task_id = stable_hash({'kind': kind, 'payload': payload, 'corpus': CORPUS})
    with conn:
        conn.execute('INSERT OR IGNORE INTO tasks(id,kind,payload,status) VALUES(?,?,?,?)',
                     (task_id, kind, json.dumps(payload, sort_keys=True), 'pending'))
    return task_id

portfolio_tasks = {aid: add_task('portfolio', {'author_id': aid}) for aid in sorted(refresh_ids)}

class StopCollection(Exception): pass
class BadPage(Exception): pass
session = requests.Session()
requests_this_run = 0
stop_reason = 'collection_not_started'

# No key is requested or sent until the API cell is run.
def get_api_key():
    key = os.environ.get('OPENALEX_API_KEY', '').strip()
    if not key:
        key = getpass.getpass('OpenAlex API key (hidden; not saved): ').strip()
    if not key: raise StopCollection('No API key supplied.')
    return key

def request_page(params):
    global requests_this_run
    for attempt in range(3):
        used_today = conn.execute('SELECT COUNT(*) AS n FROM request_ledger WHERE utc_day=?', (day(),)).fetchone()['n']
        if requests_this_run >= MAX_REQUESTS_PER_RUN: raise StopCollection('Per-run request cap reached; resume on another run.')
        if used_today >= MAX_REQUESTS_PER_UTC_DAY: raise StopCollection('Local UTC-day request cap reached; resume after reset.')
        with conn:
            cur = conn.execute('INSERT INTO request_ledger VALUES(?,?,?)', (day(), now(), 'attempted'))
            ledger_id = cur.lastrowid
        requests_this_run += 1
        time.sleep(REQUEST_INTERVAL_SECONDS)
        try:
            response = session.get('https://api.openalex.org/works', params=params, timeout=(15, 120))
        except requests.RequestException:
            if attempt == 2: raise StopCollection('Network failure; cursor saved. No credentials logged.')
            time.sleep(2 ** attempt)
            continue
        with conn:
            conn.execute('UPDATE request_ledger SET status=? WHERE rowid=?', (str(response.status_code), ledger_id))
        if response.status_code in (401, 403, 429):
            raise StopCollection('HTTP ' + str(response.status_code) + ': check key/activation/budget/rate limit; progress saved.')
        if response.status_code >= 500:
            if attempt == 2: raise StopCollection('Repeated server error; cursor saved.')
            time.sleep(2 ** attempt); continue
        if response.status_code != 200:
            raise StopCollection('HTTP ' + str(response.status_code) + ': inspect query compatibility before resuming; response body not logged.')
        try: return response.json()
        except ValueError: raise StopCollection('Non-JSON API response; cursor unchanged.')
    raise StopCollection('Request retry limit reached.')

def validate_page(task, body):
    payload = json.loads(task['payload'])
    meta = body.get('meta')
    results = body.get('results')
    if not isinstance(meta, dict) or not isinstance(results, list): raise BadPage('missing_meta_or_results')
    total = meta.get('count')
    if isinstance(total, bool) or not isinstance(total, int) or total < 0: raise BadPage('invalid_meta_count')
    if task['expected'] is not None and total != task['expected']: raise BadPage('api_total_changed_during_pagination')
    cursor = meta.get('next_cursor')
    if cursor is not None and not isinstance(cursor, str): raise BadPage('invalid_cursor')
    if cursor == task['cursor']: raise BadPage('repeated_cursor')
    if not results and cursor: raise BadPage('empty_page_with_nonterminal_cursor')
    items = []; edges = []
    target_set = set(payload.get('targets', []))
    seen = set()
    for work in results:
        if not isinstance(work, dict) or not nid(work.get('id')): raise BadPage('invalid_result_work_id')
        wid = nid(work['id']); year = yr(work.get('publication_year'))
        if wid in seen: raise BadPage('duplicate_work_within_page')
        seen.add(wid)
        existing = conn.execute('SELECT year FROM items WHERE task=? AND id=?', (task['id'], wid)).fetchone()
        if existing is not None: raise BadPage('duplicate_work_across_pages_or_live_graph_drift')
        if year is not None:
            recorded_years = {r['year'] for r in conn.execute('SELECT DISTINCT year FROM items WHERE id=? AND year IS NOT NULL', (wid,))}
            if recorded_years and recorded_years != {year}: raise BadPage('publication_year_conflict_across_queries')
        author_ids = []
        if task['kind'] == 'citations':
            if year is None or not WINDOW_LO <= year <= WINDOW_HI: raise BadPage('invalid_or_out_of_window_citing_year')
            references = work.get('referenced_works')
            authorships = work.get('authorships')
            if not isinstance(references, list) or not isinstance(authorships, list): raise BadPage('missing_references_or_authorships')
            if any(nid(x) is None for x in references): raise BadPage('invalid_reference_id')
            matched = {nid(x) for x in references} & target_set
            if not matched: raise BadPage('no_target_reference_overlap_possible_merged_ids')
            for slot in authorships:
                a = slot.get('author') if isinstance(slot, dict) else None
                aid = nid(a.get('id'), 'A') if isinstance(a, dict) else None
                if aid: author_ids.append(aid)
            edges.extend((task['id'], wid, cited, year) for cited in sorted(matched))
        items.append((task['id'], wid, year, json.dumps(sorted(set(author_ids)))))
    return total, cursor, items, edges

def run_task(task_id):
    while True:
        task = conn.execute('SELECT * FROM tasks WHERE id=?', (task_id,)).fetchone()
        if task['status'] in ('complete', 'blocked'): return
        p = json.loads(task['payload'])
        if task['kind'] == 'portfolio':
            filter_value = 'author.id:' + p['author_id']
            select = 'id,publication_year'
        else:
            filter_value = 'cites:' + '|'.join(p['targets']) + ',publication_year:' + str(WINDOW_LO) + '-' + str(WINDOW_HI)
            select = 'id,publication_year,referenced_works,authorships'
        body = request_page({'filter': filter_value, 'select': select, 'per_page': PER_PAGE,
                             'cursor': task['cursor'], 'corpus': CORPUS})
        try:
            total, cursor, items, edges = validate_page(task, body)
            collected = conn.execute('SELECT COUNT(*) AS n FROM items WHERE task=?', (task_id,)).fetchone()['n'] + len(items)
            if collected > total: raise BadPage('more_unique_results_than_meta_count')
            if cursor is None and collected != total: raise BadPage('terminal_page_count_mismatch')
            with conn:
                conn.executemany('INSERT INTO items VALUES(?,?,?,?)', items)
                conn.executemany('INSERT OR IGNORE INTO edges VALUES(?,?,?,?)', edges)
                conn.execute('UPDATE tasks SET expected=?,cursor=?,pages=pages+1,status=?,started=COALESCE(started,?),finished=? WHERE id=?',
                    (total, cursor or '', 'complete' if cursor is None else 'running', now(), now() if cursor is None else None, task_id))
        except BadPage as exc:
            with conn:
                conn.execute("UPDATE tasks SET status='blocked',issue=? WHERE id=?", (str(exc), task_id))
            raise StopCollection('A query was blocked by data-validation checks. See query_progress.csv; do not replace missing counts with zero.')

## 3. Refresh flagged portfolios first, then lock the target plan

Newly collected portfolios are kept separate; steps 3/4 are NOT overwritten. Full successful cursor collection establishes API-query coverage at collection time, not author-ID disambiguation. Reference award papers remain known targets even if absent from the refreshed portfolio; that disagreement is flagged.

Only after every planned portfolio refresh completes is the shared citation-batch plan created. Stop/resume leaves all other author rows visible with missing counts. Unflagged legacy portfolios remain unverified even when their citation batches finish.

In [4]:
def assemble_targets():
    statuses = {r['id']: r['status'] for r in conn.execute('SELECT id,status FROM tasks')}
    refresh_complete = all(statuses[t] == 'complete' for t in portfolio_tasks.values())
    mapping = {}; portfolio_rows = []
    for a in master4.itertuples(index=False):
        aid = a.author_id
        if aid in portfolio_tasks and statuses[portfolio_tasks[aid]] == 'complete':
            ids = {r['id'] for r in conn.execute('SELECT id FROM items WHERE task=?', (portfolio_tasks[aid],))}
            source = 'refreshed_api_portfolio'
            verified_query = True
        else:
            ids = set(legacy.get(aid, set()))
            source = 'legacy_cache_unverified'
            verified_query = False
        missing_ref = expected_refs[aid] - ids
        known = ids | expected_refs[aid]
        mapping[aid] = known
        portfolio_rows.append({'author_id': aid, 'portfolio_source': source, 'portfolio_api_pagination_complete': verified_query,
            'n_portfolio_targets': len(known), 'n_reference_works_added': len(missing_ref),
            'reference_work_missing_from_portfolio_query': bool(missing_ref),
            'refresh_pending': aid in portfolio_tasks and not verified_query,
            'citation_portfolio_identity_independently_verified': False})
    plan = []
    if refresh_complete:
        unique_ids = sorted(set().union(*mapping.values()))
        for start in range(0, len(unique_ids), BATCH_SIZE):
            payload = {'targets': unique_ids[start:start+BATCH_SIZE], 'year_lo': WINDOW_LO, 'year_hi': WINDOW_HI}
            tid = add_task('citations', payload)
            plan.append((tid, payload['targets']))
        fingerprint = stable_hash(sorted((a, sorted(ids)) for a, ids in mapping.items()))
        old = conn.execute("SELECT value FROM metadata WHERE key='target_plan_hash'").fetchone()
        if old and old['value'] != fingerprint: raise RuntimeError('Target plan changed. Use a new collection version after review.')
        with conn: conn.execute("INSERT OR IGNORE INTO metadata VALUES('target_plan_hash',?)", (fingerprint,))
    return mapping, pd.DataFrame(portfolio_rows), plan, refresh_complete

try:
    if RUN_API_COLLECTION:
        session.headers['Authorization'] = 'Bearer ' + get_api_key()
        try:
            for tid in portfolio_tasks.values(): run_task(tid)
            mapping, portfolio_audit, citation_plan, all_refresh_done = assemble_targets()
            if all_refresh_done:
                print('Citation batches:', len(citation_plan), '| At least this many list requests; pagination adds more.')
                for tid, targets in citation_plan: run_task(tid)
            else:
                raise StopCollection('Portfolio refresh is incomplete/blocked. See query progress before collecting citations.')
            stop_reason = 'all_current_queries_complete'
        finally:
            session.headers.pop('Authorization', None)
    else:
        stop_reason = 'preview_only_no_live_requests'
except StopCollection as exc:
    stop_reason = str(exc)
finally:
    session.headers.pop('Authorization', None)
mapping, portfolio_audit, citation_plan, all_refresh_done = assemble_targets()
print('Collection status:', stop_reason)
print('HTTP attempts this run:', requests_this_run)
print('The next run resumes saved queries; no key is stored.')

Citation batches: 3695 | At least this many list requests; pagination adds more.
Collection status: A query was blocked by data-validation checks. See query_progress.csv; do not replace missing counts with zero.
HTTP attempts this run: 190
The next run resumes saved queries; no key is stored.


In [6]:
blocked = conn.execute("""
    SELECT *
    FROM tasks
    WHERE kind = 'citations'
      AND status = 'blocked'
      AND issue = 'no_target_reference_overlap_possible_merged_ids'
""").fetchall()

assert len(blocked) == 1, (
    f"Found {len(blocked)} matching blocked queries; inspect before proceeding."
)

task = blocked[0]
payload = json.loads(task["payload"])
target_ids = set(payload["targets"])

params = {
    "filter": (
        "cites:" + "|".join(payload["targets"])
        + f",publication_year:{WINDOW_LO}-{WINDOW_HI}"
    ),
    "select": "id,publication_year,referenced_works,authorships",
    "per_page": PER_PAGE,
    "cursor": task["cursor"],
    "corpus": CORPUS,
}

try:
    session.headers["Authorization"] = "Bearer " + get_api_key()
    blocked_page_debug = request_page(params)
finally:
    session.headers.pop("Authorization", None)

unmatched = []
for work in blocked_page_debug.get("results", []):
    references = {
        nid(ref)
        for ref in work.get("referenced_works", [])
        if nid(ref)
    }
    if not references.intersection(target_ids):
        unmatched.append({
            "citing_work_id": work["id"],
            "publication_year": work.get("publication_year"),
            "reference_count": len(references),
            "sample_reference_ids": sorted(references)[:10],
        })

display(pd.DataFrame(unmatched).head(5))

,citing_work_id,publication_year,reference_count,sample_reference_ids
0,https://openalex.org/W2467203645,2015,119,"[W1485383500, W1489992831, W1545878925, W15517..."


In [9]:
import requests
import time

alias_file = STATE_DIR / "target_alias_diagnostic.json"
aliases = (
    json.loads(alias_file.read_text(encoding="utf-8"))
    if alias_file.exists()
    else {}
)

client = requests.Session()
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

client.mount(
    "https://api.openalex.org/",
    HTTPAdapter(max_retries=Retry(
        total=3,
        backoff_factor=2,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
        respect_retry_after_header=False,
    )),
)
lookup_errors = []

try:
    client.headers["Authorization"] = "Bearer " + get_api_key()

    for old_id in sorted(target_ids):
        if old_id in aliases:
            continue

        time.sleep(0.2)

        try:
            response = client.get(
                f"https://api.openalex.org/works/{old_id}",
                timeout=30,
            )
        except requests.RequestException:
            lookup_errors.append((old_id, "network_error"))
            break

        if response.status_code == 404:
            lookup_errors.append((old_id, "HTTP_404_unresolved"))
            continue  # Record the gap, but inspect the other target IDs.

        if response.status_code != 200:
            lookup_errors.append(
                (old_id, f"HTTP_{response.status_code}")
            )
            break  # Still stop for authentication, rate-limit, or server errors.

        try:
            canonical_id = nid(response.json().get("id"))
        except ValueError:
            canonical_id = None

        if canonical_id is None:
            lookup_errors.append((old_id, "invalid_response_id"))
            break

        aliases[old_id] = canonical_id
        alias_file.write_text(
            json.dumps(aliases, indent=2),
            encoding="utf-8",
        )
finally:
    client.headers.pop("Authorization", None)
    client.close()

# Compare canonical targets with the problematic citing paper's references.
problem_work = next(
    w for w in blocked_page_debug["results"]
    if nid(w["id"]) == "W2467203645"
)
problem_refs = {
    nid(ref)
    for ref in problem_work["referenced_works"]
    if nid(ref)
}

diagnosis = pd.DataFrame([
    {
        "old_target_id": old_id,
        "canonical_target_id": aliases[old_id],
        "id_changed": old_id != aliases[old_id],
        "found_in_problem_paper_references":
            aliases[old_id] in problem_refs,
    }
    for old_id in sorted(target_ids)
    if old_id in aliases
])

display(diagnosis.loc[
    diagnosis["id_changed"]
    | diagnosis["found_in_problem_paper_references"]
])
print("Targets resolved:", len(set(target_ids) & set(aliases)),
      "of", len(target_ids))
print("Lookup errors:", lookup_errors)

,old_target_id,canonical_target_id,id_changed,found_in_problem_paper_references
0,W101003559,W4246040647,True,True
91,W101830177,W4206217088,True,False


Targets resolved: 98 of 100
Lookup errors: [('W101060335', 'HTTP_404_unresolved'), ('W101505544', 'HTTP_404_unresolved')]


## 4. Aggregate edge-year counts with explicit completeness states

`citations_received_cached_observed` is the currently observed edge count and can increase on a later run. The main `citations_received` is blank until all target batches for that author finish. Then zero means zero indexed citation links returned by the completed planned queries, **not** proof of complete world citation coverage.

All portfolios/reference assignments remain provisional. A completed citation query cannot repair an incomplete legacy publication portfolio. Counts include self-citations; no self-citation classification is performed yet. Citing-author IDs are retained in the database as evidence for step 6, with no assumption that missing IDs mean non-self-citation.

In [ ]:
statuses = {r['id']: r['status'] for r in conn.execute('SELECT id,status FROM tasks')}
work_task = {wid: tid for tid, targets in citation_plan for wid in targets}
work_to_authors = defaultdict(set)
for aid, ids in mapping.items():
    for wid in ids: work_to_authors[wid].add(aid)
# Work IDs are assigned to exactly one batch. Deduplicate repeated links defensively in SQL.
edge_counts = Counter()
for row in conn.execute('SELECT cited,year,COUNT(DISTINCT citing) AS n FROM edges GROUP BY cited,year'):
    for aid in work_to_authors.get(row['cited'], set()):
        edge_counts[(aid, row['year'])] += int(row['n'])

long_rows = []; wide_rows = []
for a in master4.itertuples(index=False):
    aid = a.author_id; target_ids = mapping[aid]
    tids = {work_task[w] for w in target_ids if w in work_task}
    done = all_refresh_done and bool(target_ids) and all(w in work_task for w in target_ids) and all(statuses[t] == 'complete' for t in tids)
    coverage_row = portfolio_audit.loc[portfolio_audit.author_id.eq(aid)].iloc[0]
    wide = {'author_id': aid, 'citation_queries_complete_for_planned_targets': done,
            'citation_counts_provisional': True, 'n_citation_target_works': len(target_ids),
            'n_target_batches': len(tids), 'n_target_batches_complete': sum(statuses[t]=='complete' for t in tids)}
    for k in OFFSETS:
        y = int(a.reference_award_year) + k
        observed = int(edge_counts.get((aid,y), 0))
        main = observed if done else None
        col = 'cites_m'+str(-k) if k<0 else 'cites_p'+str(k) if k>0 else 'cites_award_year'
        wide[col] = main
        wide[col + '_observed_so_far'] = observed
        long_rows.append({'author_id': aid, 'reference_award_year': int(a.reference_award_year),
            'relative_year': k, 'calendar_year': y, 'period': 'before' if k<0 else 'after' if k>0 else 'award_year',
            'citations_received': main, 'citations_received_cached_observed': observed,
            'citation_queries_complete_for_planned_targets': done,
            'portfolio_api_pagination_complete': bool(coverage_row.portfolio_api_pagination_complete),
            'citation_counts_provisional': True, 'self_citations_included': True})
    wide_rows.append(wide)
long = pd.DataFrame(long_rows); wide = pd.DataFrame(wide_rows)
for c in ['citations_received','citations_received_cached_observed']: long[c]=long[c].astype('Int64')
cite_cols = ['cites_m'+str(-k) if k<0 else 'cites_p'+str(k) if k>0 else 'cites_award_year' for k in OFFSETS]
for c in cite_cols+[c+'_observed_so_far' for c in cite_cols]: wide[c]=wide[c].astype('Int64')
wide['cites_before_5y'] = wide[['cites_m'+str(i) for i in range(1,6)]].sum(axis=1,min_count=5).astype('Int64')
wide['cites_after_5y'] = wide[['cites_p'+str(i) for i in range(1,6)]].sum(axis=1,min_count=5).astype('Int64')
wide['cites_total_11y'] = wide[cite_cols].sum(axis=1,min_count=11).astype('Int64')
new_cols=(set(wide)|set(portfolio_audit)) & (set(master4)-{'author_id'})
if new_cols: raise ValueError('Unexpected overlapping output columns: '+repr(new_cols))
master5=master4.merge(portfolio_audit,on='author_id',validate='one_to_one').merge(wide,on='author_id',validate='one_to_one')
assert len(master5)==len(master4) and master5.author_id.is_unique
assert len(long)==len(master4)*11 and not long[['author_id','relative_year']].duplicated().any()
assert long.calendar_year.eq(long.reference_award_year+long.relative_year).all()
assert long.loc[~long.citation_queries_complete_for_planned_targets,'citations_received'].isna().all()
assert long.citations_received.dropna().ge(0).all()
for k,c in zip(OFFSETS,cite_cols):
    pd.testing.assert_series_equal(master5.set_index('author_id')[c].sort_index(),
        long.loc[long.relative_year.eq(k)].set_index('author_id').citations_received.sort_index(),check_names=False)
display(master5[['author_id','author_name','reference_award_year','citation_queries_complete_for_planned_targets',*cite_cols]].head(20))

## 5. Export an immutable progress snapshot on each run

If collection is paused, this is a **progress snapshot**, not a finished step 5. Main blank counts are intentional. Continue on later runs until progress is complete, and resolve blocked tasks before calling the dataset final.

API queries can span dates and the live graph can change. Start/end timestamps and validation checks expose some drift, but do not create an immutable historical database snapshot. Keep current cohort and portfolio limitations in final reporting. Repairs here do not silently update earlier publication/seniority files; any upstream repair must be a deliberate versioned rerun.

In [ ]:
progress = pd.read_sql_query('SELECT id,kind,status,expected,pages,issue,started,finished FROM tasks ORDER BY kind,id',conn)
local_usage = pd.read_sql_query('SELECT utc_day,COUNT(*) AS http_attempts FROM request_ledger GROUP BY utc_day',conn)
summary_values = {
    'authors_retained':len(master5), 'author_year_rows':len(long),
    'portfolio_refresh_queries_planned':len(portfolio_tasks),
    'portfolio_refresh_queries_complete':sum(statuses[t]=='complete' for t in portfolio_tasks.values()),
    'citation_batches_planned':len(citation_plan),
    'citation_batches_complete':sum(statuses[t]=='complete' for t,_ in citation_plan),
    'blocked_queries':int(progress.status.eq('blocked').sum()),
    'authors_with_complete_planned_citation_queries':int(wide.citation_queries_complete_for_planned_targets.sum()),
    'authors_still_missing_main_citation_series':int((~wide.citation_queries_complete_for_planned_targets).sum()),
    'distinct_indexed_edges_collected':conn.execute('SELECT COUNT(*) AS n FROM edges').fetchone()['n'],
    'author_year_cells_before_2012':int(history_need.historical_rollup_unsafe.sum()),
    'http_attempts_this_run':requests_this_run,
    'independently_verified_author_identities':0}
summary=pd.DataFrame(summary_values.items(),columns=['measure','value'])
display(summary)
print('Stop reason:',stop_reason)
SNAPSHOT_DIR=ROOT/'conf_paper/conf_data/derived'/COLLECTION_VERSION/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SNAPSHOT_DIR.mkdir(parents=True,exist_ok=False)
tables={'author_yearly_citations.csv':master5,'yearly_citations_long.csv':long,
        'portfolio_coverage.csv':portfolio_audit,'query_progress.csv':progress,
        'local_request_usage.csv':local_usage,'step5_summary.csv':summary,
        'historical_data_needed.csv':history_need,'existing_citation_file_inventory.csv':inventory}
for name,frame in tables.items(): frame.to_csv(SNAPSHOT_DIR/name,index=False)
conn.commit()
output_hashes={p.name:sha(p) for p in SNAPSHOT_DIR.iterdir() if p.is_file()}
meta={'created_at_utc':now(),'identity':identity,'output_sha256':output_hashes,
      'stop_reason':stop_reason,'all_planned_author_citation_queries_complete':bool(wide.citation_queries_complete_for_planned_targets.all()),
      'checkpoint_database':'conf_paper/conf_data/collection/'+COLLECTION_VERSION+'/citation_collection.sqlite',
      'credentials_saved':False,'upstream_files_modified':False,
      'definition':'distinct citing-work/cited-portfolio-work edges binned by citing publication year; self-citations included',
      'limits':['Legacy portfolio completeness remains unverified for unrefreshed authors.',
                'Known targets include reference award works even when portfolio query misses them.',
                'OpenAlex ID merges and missing references can affect coverage; not all are detectable.',
                'No absence-of-rollup-year zero imputation.', 'Current graph, not historical index snapshot.',
                'Observed-so-far zeros do not mean completed collection.',
                'Citing-author IDs retained in checkpoint are not guaranteed complete for step 6.']}
(SNAPSHOT_DIR/'manifest.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
print('Saved progress snapshot:',SNAPSHOT_DIR)
print('Main counts become available only as required citation queries finish.')